# Simple AWG Example

Generate simple `int16` DAC waveforms, preview them, load them into the DAC BRAM players, and enable or disable RF output explicitly. DAC0 uses a zero-padded full-BRAM write, then the active hardware loop length is set separately.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from firmware import OverlayController
from firmware import signals

In [ ]:
ol = OverlayController()
info = ol.info()
info

In [ ]:
DAC0_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
DAC2_SR = float(info["rfdc"]["dac2_sampling_rate_gsps"]) * 1e9
DAC0_LEN = int(info["dac0"]["bram_int16_samples"])
DAC2_LEN = int(info["dac2"]["bram_int16_samples"])
DAC_PEAK = int(0.8 * np.iinfo(np.int16).max)
SAMPLES_PER_VECTOR = 512 // 16

DAC0_SR, DAC2_SR, DAC0_LEN, DAC2_LEN, DAC_PEAK

In [ ]:
def to_int16(waveform, peak=DAC_PEAK):
    data = np.asarray(waveform, dtype=float)
    return np.round(np.clip(data, -float(peak), float(peak))).astype(np.int16)


def samples_for_duration(duration_s, sample_rate_hz, player):
    samples = int(round(float(duration_s) * float(sample_rate_hz)))
    if samples <= 0:
        raise ValueError("duration must produce at least one sample")
    if samples > player.capacity:
        raise ValueError(
            f"duration requires {samples} samples, but {player.name} capacity is {player.capacity}"
        )
    return samples


def duration_for_samples(num_samples, sample_rate_hz):
    return int(num_samples) / float(sample_rate_hz)


def vectors_for_samples(num_samples):
    return int(np.ceil(int(num_samples) / SAMPLES_PER_VECTOR))


def read_dac0_length_raw():
    return int(ol.gpio_control.axi_gpio_dac.mmio.read(0x8)) & ol.dac0.length_mask


def plot_waveform(waveform, sample_rate, samples=2048, title="Waveform"):
    view = np.asarray(waveform)[:samples]
    time_ns = np.arange(view.size) / sample_rate * 1e9

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time_ns, view)
    ax.set_title(title)
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("DAC code")
    ax.grid(True, alpha=0.3)
    return ax


def plot_spectrum(waveform, sample_rate, title="Spectrum"):
    data = np.asarray(waveform, dtype=float)
    spectrum = np.fft.rfft(data)
    freqs_mhz = np.fft.rfftfreq(data.size, d=1 / sample_rate) / 1e6
    magnitude_db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-12))

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(freqs_mhz, magnitude_db)
    ax.set_title(title)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.grid(True, alpha=0.3)
    return ax

## Generate Waveforms

The signal helpers return floating-point amplitudes. Here the amplitude is already in DAC-code units, then clipped and rounded to `int16`. The active DAC0 sinewave is copied into a zero-padded full-BRAM buffer before programming.

In [ ]:
requested_total_seconds = 1.0e-6

active_samples = samples_for_duration(requested_total_seconds, DAC0_SR, ol.dac0)
actual_total_seconds = duration_for_samples(active_samples, DAC0_SR)

active_dac0_waveform = to_int16(
    signals.sine(freq_hz=100e6, sample_rate=DAC0_SR, num_samples=active_samples, amplitude=DAC_PEAK)
)
dac0_waveform = np.zeros(DAC0_LEN, dtype=np.int16)
dac0_waveform[:active_samples] = active_dac0_waveform

dac2_waveform = to_int16(
    signals.sawtooth(freq_hz=50e6, sample_rate=DAC2_SR, num_samples=DAC2_LEN, amplitude=DAC_PEAK)
)

print(f"Requested DAC0 sine length: {requested_total_seconds * 1e6:.6f} us")
print(f"Active DAC0 sine length: {active_samples} samples")
print(f"Active DAC0 vectors: {vectors_for_samples(active_samples)} vectors")
print(f"DAC0 BRAM write length: {len(dac0_waveform)} samples")
print(f"Actual DAC0 sine length: {actual_total_seconds * 1e6:.6f} us")

active_dac0_waveform.dtype, active_dac0_waveform.shape, dac0_waveform.shape, dac2_waveform.dtype, dac2_waveform.shape

In [ ]:
plot_waveform(active_dac0_waveform, DAC0_SR, title="DAC0 active waveform")
plot_spectrum(active_dac0_waveform, DAC0_SR, title="DAC0 active spectrum")

plot_waveform(dac2_waveform, DAC2_SR, title="DAC2 waveform")
plot_spectrum(dac2_waveform, DAC2_SR, title="DAC2 spectrum");

## Program DAC Players

Disable the outputs before programming. DAC0 writes the full zero-padded BRAM buffer first, then sets the active loop length.

In [ ]:
print("Before programming")
print(f"  active_samples: {active_samples} ({active_samples:#x})")
print(f"  active vectors: {vectors_for_samples(active_samples)}")
print(f"  DAC0 BRAM samples to write: {len(dac0_waveform)} ({len(dac0_waveform):#x})")
print(f"  DAC2 BRAM samples to write: {len(dac2_waveform)} ({len(dac2_waveform):#x})")
print(f"  raw DAC0 length before: {read_dac0_length_raw()} ({read_dac0_length_raw():#x})")

ol.dac0.disable()
ol.dac2.disable()
time.sleep(0.01)

ol.dac0.load_waveform(dac0_waveform)
print(f"Raw DAC0 length after BRAM load: {read_dac0_length_raw()} ({read_dac0_length_raw():#x})")

ol.dac0.set_waveform_length(active_samples)
time.sleep(0.01)
length_readback = read_dac0_length_raw()
print(f"Raw DAC0 length after active length write: {length_readback} ({length_readback:#x})")

if length_readback != active_samples:
    raise RuntimeError(
        f"DAC0 length mismatch: wrote {active_samples:#x}, read {length_readback:#x}"
    )

ol.dac2.load_waveform(dac2_waveform)

ol.info()

## Enable Outputs

Only run this cell when the RF chain and instruments are ready.

In [ ]:
ol.dac0.enable()
ol.dac2.enable()

ol.dac0.is_enabled(), ol.dac2.is_enabled()

## Disable Outputs

Run this before changing cabling or loading a different waveform.

In [ ]:
ol.dac0.disable()
ol.dac2.disable()

ol.info()